In [0]:

bronze = "abfss://bronze@databricktraveljournal.dfs.core.windows.net"
table = "accounts"
bronze_table_parquet_path = f"{bronze}/{table}"

account_df = spark.read.format("parquet")\
    .load(f"{bronze_table_parquet_path}")

display(account_df)

In [0]:
account_df.printSchema()


In [0]:
account_df.display()

### Quality

In [0]:
from pyspark.sql import DataFrame
from pyspark.sql import functions as F
from pyspark.sql.functions import col, coalesce, get_json_object, to_timestamp, lit, when


def transform_to_silver(bronze_df: DataFrame) -> DataFrame:
    df = bronze_df



    # date_type is a real date; created_at is "-" so we skip it
    df = df.withColumn("created_at",
                               when(col("created_at").isNull(),
                                    get_json_object(col("_rescued_data"),"$.created_at"))
                                    .otherwise(col("created_at"))
                                    )
    df = (
        df
        # if the year is 2024, replace it with 2026 (keeps month/day/time exactly)
        .withColumn(
            "created_at",
            F.when(
                F.col("created_at").startswith("2024"),
                F.regexp_replace("created_at", r"^2024", "2026")
            ).otherwise(F.col("created_at"))
        )
        .withColumn("created_at", F.to_timestamp("created_at"))
        .withColumn("date_type", F.to_date("created_at"))
    )
    df = (
        df.withColumn("date_type", F.to_date("date_type"))
        .withColumn("created_at", F.to_timestamp("created_at"))
        .withColumn("last_login", F.to_timestamp("last_login"))

        )

    df = df.withColumn("flag", F.lower(F.trim(F.col("flag"))) == F.lit("true"))


    df = (
        df.withColumn("year",          F.expr("try_cast(year as int)"))
          .withColumn("month",         F.expr("try_cast(month as int)"))
          .withColumn("day",           F.expr("try_cast(day as int)"))
          .withColumn("user_id",            F.expr("try_cast(user_id as long)"))
          .withColumn("image_id", F.expr("try_cast(image_id as int)"))   # numeric FK
          .withColumn("year",  F.when(F.col("year") == 2024, F.lit(2026)).otherwise(F.col("year")))

    )

    df = (
        df.withColumn("email",    F.trim(F.col("email")))
          .withColumn("username", F.trim(F.col("username")))
          .withColumn("role",     F.trim(F.col("role")))
        
    )

    email_pattern = r"^[^@\s]+@[^@\s]+\.[^@\s]+$"
    
    df = df.withColumn(
        "_is_valid",
        coalesce(
            col("user_id").isNotNull()
            & col("username").isNotNull()
            & col("email").isNotNull()
            & col("email").rlike(email_pattern),
            lit(False),
        ),
    )

    valid_df   = df.filter(col("_is_valid"))
    invalid_df = df.filter(~col("_is_valid"))

    invalid_count = invalid_df.count()
    if invalid_count > 0:
        print(f"Quarantined {invalid_count} invalid records")

    return valid_df.drop("_rescued_data","_is_valid")


df = transform_to_silver(account_df)
   

In [0]:
df.display()

### Deduplicate

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

def deduplicate_by_key(df, keycolumns, order_column, ascending=False):
    """
    
    Deduplicate a Dataframe by composite key, keeping the row with the highest (or lowest) value in order_column

    Args:
        df: Input DataFrame with duplicates
        key_columns: List of columns forming the composite key
        order_column: Column to break ties (e.g., updated_at)
        ascending: If True, keep the smallest order_column value
    """
    
    order_expr = (
        F.col(order_column).asc() if ascending else F.col(order_column).desc()
    )

    window_spec = Window.partitionBy(*keycolumns).orderBy(order_expr)

    return df.withColumn("rank", F.row_number().over(window_spec)).filter(
        F.col("rank") == 1
    ).drop("rank")

account_df = deduplicate_by_key(df, ["user_id"], "created_at", ascending=True)



In [0]:
account_df.display()

# Data Writing

In [0]:
account_df.write.format("delta").mode("overwrite").save("abfss://silver@databricktraveljournal.dfs.core.windows.net/accounts")

## DELTA

In [0]:
%sql

CREATE TABLE IF NOT EXISTS travel_journal_catalog.silver.accounts 
USING DELTA
LOCATION "abfss://silver@databricktraveljournal.dfs.core.windows.net/accounts"

In [0]:
%sql
SELECT * FROM travel_journal_catalog.silver.accounts